sklearn
- https://scikit-learn.org/stable/

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd

a = load_iris()
print(a.keys())
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df = pd.DataFrame(a.data, columns=feature_names)
df['species'] = a.target
df['name'] = a.target_names[a.target]
df.head(3)

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])


,sepal_length,sepal_width,petal_length,petal_width,species,name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa


In [ ]:
df.describe()

,sepal_length,sepal_width,petal_length,petal_width,species
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


irist 데이터
- 150개 붓꽃 데이터
- 50개씩 3개 품종에 대한 데이터
- feature : sepal_length, sepal_width, petal_lenth, petal_with
- target : 붓꽃의 품종 (setosa, versicolor, virginica)

In [ ]:
df.columns
df.groupby('name')[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].mean()

,sepal_length,sepal_width,petal_length,petal_width
name,,,,
setosa,5.006,3.428,1.462,0.246
versicolor,5.936,2.770,4.260,1.326
virginica,6.588,2.974,5.552,2.026


Oneway ANOVA (일원분산분석)
- 분산을 사용하여 그룹간 평균의 차이를 검정
- 그룹간분산과 그룹내분산을 사용해 F통계량을 구하고, F통계량으로 p-value를 구함(F분포)
- 귀무가설 : 그룹별 평균이 모두 동일하다.
- 대립가설 : 그룹의 평균 중 하나라도 차이가 있다.
- p-value : 1종 오류의 발생 확률
- 1종 오류 : 귀무가설을 기각하고 대립가설을 채택했을 때 발생하는 오류의 확률. (귀무가설이 참인데 기각해서 발생)
- 유의 수준 : 1종 오류의 최대 허용 한계
- p-value <= 유의 수준 : 대립가설 채택(통계적으로 유이하다, 통계적으로 유의미한 차이가 있다.)

In [ ]:
# 범주별 평균의 차이 검정
from scipy.stats import f_oneway
#x = 'petal_width'
#x = 'sepal_width'
x = 'sepal_length'

setosa = df.loc[df['name'] == 'setosa', x]
versicolor = df.loc[df['name'] == 'versicolor', x]
virginica = df.loc[df['name'] == 'virginica', x]
_, pvalue = f_oneway(setosa, versicolor, virginica)
print(pvalue, f'{pvalue:.6f}')

1.6696691907693826e-31 0.000000


In [ ]:
# 상관관계 분석
df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']].corr()

,sepal_length,sepal_width,petal_length,petal_width,species
sepal_length,1.000000,-0.117570,0.871754,0.817941,0.782561
sepal_width,-0.117570,1.000000,-0.428440,-0.366126,-0.426658
petal_length,0.871754,-0.428440,1.000000,0.962865,0.949035
petal_width,0.817941,-0.366126,0.962865,1.000000,0.956547
species,0.782561,-0.426658,0.949035,0.956547,1.000000


- X(Feature), y(Target) 데이터 분리
- train, test 데이터 분리

In [ ]:
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = df['species']

from sklearn.model_selection import train_test_split

temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = temp
print([x.shape for x in temp])

[(105, 4), (45, 4), (105,), (45,)]


In [ ]:
print(type(y_train), type(y_test))

<class 'pandas.core.series.Series'> <class 'pandas.core.series.Series'>


In [ ]:
print(y.value_counts(), y_train.value_counts(), y_test.value_counts(), sep='n')

species
0    50
1    50
2    50
Name: count, dtype: int64nspecies
1    35
0    35
2    35
Name: count, dtype: int64nspecies
2    15
1    15
0    15
Name: count, dtype: int64


분류 모델링(Modeling)
- 선형 모델 : 선을 이용해서 데이터를 분류     
- 트리 모델 : 질문을 만들어 데이터를 쪼개는 작업을 통해 분류   
- kNN 모델 : k개의 가까운 데이터를 보고 가장 빈도가 높은 범주로 분류
- 앙상블 모델 : 여러 개의 모델을 사용한 학습을 통해 분류, 다양한 종류의 앙상블 모델이 있음

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

model1 = LogisticRegression(max_iter=2000)
model1.fit(X_train, y_train)
a = model1.score(X_train, y_train)
b = model1.score(X_test, y_test)

print(f'Logistic: {a:.4f}, {b:.4f}')

model2 = DecisionTreeClassifier()
model2.fit(X_train, y_train)
a = model2.score(X_train, y_train)
b = model2.score(X_test, y_test)

print(f'DecisionTree: {a:.4f}, {b:.4f}')

model3 = KNeighborsClassifier()
model3.fit(X_train, y_train)
a = model3.score(X_train, y_train)
b = model3.score(X_test, y_test)
print(f'KNN: {a:.4f}, {b:.4f}')

Logistic: 0.9714, 0.9333
DecisionTree: 1.0000, 0.9778
KNN: 0.9714, 0.9778


kNN 모델
- 거리 기반의 모델로 Feature의 단위가 차이가 있는 경우 결과에 영향을 줄 수 있음
- 사전에 Scaling을 하여 영향을 없앨 수 있음
- 이웃의 개수(k): 하이퍼파라미터 - 사용자가 값을 지정해 성능을 최적화 할 수 있음

**min-max scaler(= min max normalization)**
- 값의 범위를 0 ~ 1로 조정
- (x_i - x_min) / (x_max - x_min)

In [ ]:
a = np.array([1, 3, 5, 7, 9])
print((a - a.min()) / (a.max() - a.min()))

[0.   0.25 0.5  0.75 1.  ]


In [ ]:
# Scaler의 transform()을 수행해 얻는 결과는 ndarray 객체이다.
from sklearn.preprocessing import MinMaxScaler # min = 0, max = 1
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
#scaler.fit(X_train)
X_test_scaled = scaler.transform(X_test)
print(type(X_train_scaled),type(X_test_scaled))

X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)

# min=0, max=1인 것을 확인 할 수 있음
print(X_train_scaled.min(), X_train_scaled.max())

# 0과 1이 아닌 것을 확인 할 수 있음
print(X_test_scaled.min(), X_test_scaled.max())


<class 'numpy.ndarray'> <class 'numpy.ndarray'>
sepal_length    0.0
sepal_width     0.0
petal_length    0.0
petal_width     0.0
dtype: float64 sepal_length    1.0
sepal_width     1.0
petal_length    1.0
petal_width     1.0
dtype: float64
sepal_length    0.027778
sepal_width     0.125000
petal_length   -0.017241
petal_width     0.041667
dtype: float64 sepal_length    0.833333
sepal_width     0.833333
petal_length    0.896552
petal_width     0.958333
dtype: float64


In [ ]:
# MinMaxScaler를 적용한 결과가 항상 좋은 것은 아님
# 성능은 데이터의 상태에 따라 달라지며, kNN 모델에서 k가 매우 적은 경우 과대적합의 위험이 있음
from sklearn.neighbors import KNeighborsClassifier
for k in range(1, 8):
  model3 = KNeighborsClassifier(k)
  model3.fit(X_train_scaled, y_train)
  a = model3.score(X_train_scaled, y_train)
  b = model3.score(X_test_scaled, y_test)
  print(f'KNeighbors: {a:.4f}, {b:.4f}')

KNeighbors: 1.0000, 0.9333
KNeighbors: 0.9810, 0.9111
KNeighbors: 0.9810, 0.9333
KNeighbors: 0.9905, 0.9111
KNeighbors: 0.9810, 0.9333
KNeighbors: 0.9810, 0.9333
KNeighbors: 0.9714, 0.9333


In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

a = load_iris()
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df = pd.DataFrame(a.data, columns=feature_names)
df['species'] = a.target

X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = df['species']
temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = temp
model3 = KNeighborsClassifier()
model3.fit(X_train, y_train)
a = model3.score(X_train, y_train)
b = model3.score(X_test, y_test)
print(f'KNeighbors: {a:.4f}, {b:.4f}')

KNeighbors: 0.9714, 0.9778


예측 값, 예측 확률 구하기

In [ ]:
result = pd.DataFrame()
result['실제값'] = y_test.to_list()
result['예측값'] = model3.predict(X_test)
result['정답여부'] = result['실제값'] == result['예측값']
print(result)

    실제값  예측값   정답여부
0     2    2   True
1     1    1   True
2     2    2   True
3     1    1   True
4     2    2   True
5     2    2   True
6     1    1   True
7     1    1   True
8     0    0   True
9     2    2   True
10    0    0   True
11    0    0   True
12    2    2   True
13    2    2   True
14    0    0   True
15    2    2   True
16    1    1   True
17    0    0   True
18    0    0   True
19    0    0   True
20    1    1   True
21    0    0   True
22    1    1   True
23    2    2   True
24    2    2   True
25    1    1   True
26    1    1   True
27    1    1   True
28    1    1   True
29    0    0   True
30    2    2   True
31    2    2   True
32    1    1   True
33    0    0   True
34    2    2   True
35    0    0   True
36    0    0   True
37    0    0   True
38    0    0   True
39    1    1   True
40    1    1   True
41    0    0   True
42    2    1  False
43    2    2   True
44    1    1   True


In [ ]:
proba = pd.DataFrame(model3.predict_proba(X_test), columns=['0일 확률', '1일 확률', '2일 확률'])

print(proba.tail())

    0일 확률  1일 확률  2일 확률
40    0.0    1.0    0.0
41    1.0    0.0    0.0
42    0.0    0.8    0.2
43    0.0    0.0    1.0
44    0.0    1.0    0.0


### 유방암 데이터(2진분류)
- 양성, 음성의 두가지 범주

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler

a = load_breast_cancer()
feature = ['worst concave points',
            'worst area',
            'worst texture']

df = pd.DataFrame(a.data, columns = a.feature_names)
df['class'] = a.target
print(feature)
X = df[feature]
y = df['class']

temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = temp

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(type(X_train_scaled), type(X_test_scaled))

X_train_scaled = pd.DataFrame(X_train_scaled,
                              index=X_train.index,
                              columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled,
                              index=X_test.index,
                              columns=X_test.columns)
for k in range(1,10):
    model3 = KNeighborsClassifier(k)
    model3.fit(X_train_scaled, y_train)
    a = model3.score(X_train_scaled, y_train)
    b = model3.score(X_test_scaled, y_test)
    print(f'KNeighbors_{k}: {a:.4f}, {b:.4f}')


['worst concave points', 'worst area', 'worst texture']
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
KNeighbors_1: 1.0000, 0.9649
KNeighbors_2: 0.9824, 0.9415
KNeighbors_3: 0.9799, 0.9649
KNeighbors_4: 0.9749, 0.9708
KNeighbors_5: 0.9749, 0.9708
KNeighbors_6: 0.9774, 0.9649
KNeighbors_7: 0.9749, 0.9649
KNeighbors_8: 0.9774, 0.9532
KNeighbors_9: 0.9749, 0.9708


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler

a = load_breast_cancer()
df = pd.DataFrame(a.data, columns= a.feature_names)
df['class'] = a.target

X = df[['worst area', 'worst texture', 'worst concave points']]
y = df['class']
temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = temp

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled,
                             index=X_train.index,
                             columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled,
                             index=X_test.index,
                             columns=X_test.columns)


for k in range(1,6):
  model = KNeighborsClassifier(k)
  model.fit(X_train_scaled, y_train)
  a = model.score(X_train_scaled, y_train)
  b = model.score(X_test_scaled, y_test)
  print(f'KNeighbors: {a:.4f}, {b:.4f}')

# # 중요도 검사
# model = DecisionTreeClassifier()
# model.fit(X_train, y_train)
# temp = pd.Series(model.feature_importances_, index=X_train.columns)
# print(temp)

# print(df.columns)
# df.head()

# worst radius               0.712322
# worst concave points       0.128755
# worst texture              0.046323
# texture error              0.030071
# worst concavity            0.018606

KNeighbors: 1.0000, 0.9649
KNeighbors: 0.9774, 0.9415
KNeighbors: 0.9774, 0.9708
KNeighbors: 0.9774, 0.9649
KNeighbors: 0.9774, 0.9825


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

a = load_breast_cancer()
df = pd.DataFrame(a.data, columns=a.feature_names)
df['target'] = a.target

X = df[a.feature_names]
y = df['target']

temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = temp
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)

for k in range(1, 8):
  model3 = KNeighborsClassifier(k)
  model3.fit(X_train_scaled, y_train)
  a = model3.score(X_train_scaled, y_train)
  b = model3.score(X_test_scaled, y_test)
  print(f'KNeighbors: {a:.4f}, {b:.4f}')

KNeighbors: 1.0000, 0.9591
KNeighbors: 0.9774, 0.9591
KNeighbors: 0.9724, 0.9532
KNeighbors: 0.9824, 0.9532
KNeighbors: 0.9724, 0.9591
KNeighbors: 0.9724, 0.9591
KNeighbors: 0.9724, 0.9649


In [ ]:
from sklearn.datasets import load_breast_cancer

a = load_breast_cancer()
df = pd.DataFrame(a.data, columns=a.feature_names)
df['class'] = a.target
#print(df.columns)
#df.head()
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = temp
model2 = DecisionTreeClassifier(max_depth=3)
model2.fit(X_train, y_train)
a = model2.score(X_train, y_train)
b = model2.score(X_test, y_test)
print(f'Decision: {a:.4f}, {b:.4f}')

Decision: 0.9799, 0.9240


In [ ]:
a.keys()
a.target_names

AttributeError: 'float' object has no attribute 'keys'

In [ ]:
df['class'].value_counts()  # 0 - malignant(악성), 1 - benign(양성)

,count
class,
1,357
0,212


In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import plotly.express as px

# 데이터 로드
a = load_iris()
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df = pd.DataFrame(a.data, columns=feature_names)
df['species'] = a.target_names[a.target]

# 그래프
fig = px.scatter(
    df,
    x='petal_length',
    y='petal_width',
    color='species',
    title='Iris Scatter (Hover 가능)'
)

fig.show()

## Linear Model

In [1]:
from sklearn.datasets import fetch_openml
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

boston = fetch_openml(name='boston', version=1, as_frame=True)
df = boston.frame
df = df.astype('int')
#X = df.loc[:, 'CRIM':'LSTAT']
X = df.iloc[:, :-1]
y = df['MEDV']
temp = train_test_split(X, y, random_state=1234)
X_train, X_test, y_train, y_test = temp
print([x.shape for x in temp])
model1 = LinearRegression()
model1.fit(X_train, y_train)
a = model1.score(X_train, y_train)
b = model1.score(X_test, y_test)
print(f'Linear: {a:.4f}, {b:.4f}')


[(379, 13), (127, 13), (379,), (127,)]
Linear: 0.6917, 0.7042


PolynomialFeatures - 다항 변환


In [6]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

poly = PolynomialFeatures(degree=2)
x = np.arange(10).reshape(-1, 2)
pd.DataFrame(poly.fit_transform(x), columns=poly.get_feature_names_out())

,1,x0,x1,x0^2,x0 x1,x1^2
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0
3,1.0,6.0,7.0,36.0,42.0,49.0
4,1.0,8.0,9.0,64.0,72.0,81.0


In [12]:
from sklearn.datasets import fetch_openml
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

boston = fetch_openml(name='boston', version=1, as_frame=True)
df = boston.frame
df = df.astype('int')
X = df.iloc[:, :-1]
y = df['MEDV']

poly = PolynomialFeatures(degree=2)
x_poly = poly.fit_transform(X)
y_poly = poly.fit_transform(y.values.reshape(-1, 1))

temp = train_test_split(x_poly, y, random_state=1234)
X_train, X_test, y_train, y_test = temp
print([x.shape for x in temp])
model1 = LinearRegression()
model1.fit(X_train, y_train)
a = model1.score(X_train, y_train)
b = model1.score(X_test, y_test)
print(f'Linear: {a:.4f}, {b:.4f}')


[(379, 105), (127, 105), (379,), (127,)]
Linear: 0.8889, 0.8276


In [7]:
print(dir(poly))

['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__str__', '__subclasshook__', '__weakref__', '_build_request_for_signature', '_check_feature_names', '_check_n_features', '_combinations', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_get_default_requests', '_get_doc_link', '_get_metadata_request', '_get_param_names', '_get_tags', '_max_degree', '_min_degree', '_more_tags', '_n_out_full', '_num_combinations', '_parameter_constraints', '_repr_html_', '_repr_html_inner', '_repr_mimebundle_', '_sklearn_auto_wrap_output_keys', '_validate_data', '_validate_params', 'degree', 'fit', 'fit_transform', 'get_feature_names_out', '

,주문일시,주문번호,매출구분명,결제수단,공급가액,부가세,합계
0,2021-01-01,B0UG00RHPT,기타매출,쿠폰,909 원,91 원,"1,000 원"
1,2021-01-01,B0UG00U5GK,기타매출,회원포인트,636 원,64 원,700 원
2,2021-01-01,B0UG00XD3U,기타매출,쿠폰,909 원,91 원,"1,000 원"
3,2021-01-01,B0UG00XD3U,기타매출,회원포인트,273 원,27 원,300 원
4,2021-01-01,B0UG01BFVX,기타매출,휴대폰,"10,636 원","1,064 원","11,700 원"
